# 2D Allen–Cahn Initial-Condition Recovery

This notebook studies a nonlinear PDE inverse problem based on the Allen–Cahn equation.

We generate structured 2D initial conditions, evolve them with an exact semi-implicit Allen–Cahn solver, and observe only the later state. The inverse task is to recover the unknown initial condition from that later state.

The experiment follows the same structure as the Heat and Darcy benchmarks:

1. generate synthetic data,
2. train a forward surrogate and inverse proposal network,
3. compare D-IPG against standard optimization baselines,
4. evaluate the effect of JCP regularization through `+JCP / -JCP` ablations.

The primary metric is recovery error of the unknown initial condition.

## PDE and inverse problem

We consider the 2D Allen–Cahn equation

$$
u_t = \varepsilon^2 \Delta u + u - u^3
$$

on a periodic square domain, where:
- $u(x,y,t)$ is the state,
- $\varepsilon$ controls interface thickness.

For each sample:
- the unknown is the initial condition $u_0(x,y)$,
- the observation is the state at time $T$, denoted $u_T(x,y)$.

This produces a field-to-field inverse problem:
- unknown: $x = u_0 \in \mathbb{R}^{1 \times n_x \times n_y}$,
- observation: $y = u_T \in \mathbb{R}^{1 \times n_x \times n_y}$.

In [ ]:
import math
import time
import random
from dataclasses import dataclass, asdict
from typing import Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.func import jvp

In [ ]:
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
@dataclass
class AllenCahn2DConfig:
    nx: int = 32
    ny: int = 32

    x_channels: int = 1
    y_channels: int = 1

    epsilon: float = 0.04
    final_time: float = 0.12
    dt: float = 0.002

    noise_std: float = 0.002

    n_train: int = 2000
    n_val: int = 240
    n_test: int = 80

    hidden_channels: int = 48
    num_res_blocks: int = 4
    negative_slope: float = 0.10

    batch_size: int = 64

    forward_epochs: int = 120
    reverse_epochs: int = 80
    reverse_jcp_epochs: int = 40

    forward_lr: float = 2e-3
    reverse_lr: float = 1e-3
    reverse_jcp_lr: float = 5e-4

    forward_weight_decay: float = 1e-6
    reverse_weight_decay: float = 1e-6
    reverse_jcp_weight_decay: float = 1e-6

    reconstruction_weight: float = 1.0
    cycle_weight: float = 0.05
    probe_jcp_weight: float = 0.01

    jcp_num_probes_train: int = 1
    jcp_num_probes_eval: int = 2
    y_tilde_noise: float = 0.01
    gradient_clip: float = 5.0
    jcp_batch_subsample: int = 8

    recon_guard_factor: float = 1.005

    max_iterations: int = 80
    tolerance_eps: float = 0.12
    armijo_c: float = 1e-4
    max_backtracking_steps: int = 8
    rho: float = 0.4

    dipg_step_size: float = 1.0
    gd_step_size: float = 1.0
    gn_step_size: float = 1.0
    lm_step_size: float = 1.0
    lm_damping: float = 1e-2

    lbfgs_lr: float = 1.0
    lbfgs_max_iter: int = 80
    lbfgs_history_size: int = 20
    lbfgs_tolerance_grad: float = 1e-7
    lbfgs_tolerance_change: float = 1e-9

    cg_max_iter: int = 30
    cg_tol: float = 1e-6

    eval_subset_rjcp: int = 24
    success_rmse_threshold: float = 0.10

cfg = AllenCahn2DConfig()
print(asdict(cfg))

In [ ]:
def solve_allen_cahn_final_state(u0: np.ndarray, cfg: AllenCahn2DConfig) -> np.ndarray:
    nx, ny = u0.shape
    u = u0.astype(np.float64).copy()

    kx = 2.0 * np.pi * np.fft.fftfreq(nx, d=1.0 / nx)
    ky = 2.0 * np.pi * np.fft.fftfreq(ny, d=1.0 / ny)
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    lap = KX**2 + KY**2

    n_steps = int(round(cfg.final_time / cfg.dt))

    denom = 1.0 + cfg.dt * (cfg.epsilon ** 2) * lap

    for _ in range(n_steps):
        nonlinear = u - u**3
        rhs = u + cfg.dt * nonlinear
        rhs_hat = np.fft.fft2(rhs)
        u_hat_next = rhs_hat / denom
        u = np.fft.ifft2(u_hat_next).real

    return np.clip(u, -1.0, 1.0).astype(np.float32)

In [ ]:
def sample_initial_condition(nx: int, ny: int) -> np.ndarray:
    yy, xx = np.mgrid[0:nx, 0:ny]
    xx = (xx + 0.5) / ny
    yy = (yy + 0.5) / nx

    field = -np.ones((nx, ny), dtype=np.float32)
    family = np.random.choice(["blobs", "fronts", "rings", "mixed"])

    if family in ["blobs", "mixed"]:
        n_objects = np.random.randint(2, 6)
        for _ in range(n_objects):
            shape_type = np.random.choice(["circle", "ellipse", "rectangle"])
            cx = np.random.uniform(0.15, 0.85)
            cy = np.random.uniform(0.15, 0.85)

            if shape_type == "circle":
                r = np.random.uniform(0.05, 0.16)
                mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
            elif shape_type == "ellipse":
                rx = np.random.uniform(0.05, 0.18)
                ry = np.random.uniform(0.05, 0.18)
                theta = np.random.uniform(0.0, 2.0 * math.pi)
                x0 = xx - cx
                y0 = yy - cy
                xr = np.cos(theta) * x0 + np.sin(theta) * y0
                yr = -np.sin(theta) * x0 + np.cos(theta) * y0
                mask = (xr / rx) ** 2 + (yr / ry) ** 2 <= 1.0
            else:
                wx = np.random.uniform(0.07, 0.22)
                wy = np.random.uniform(0.07, 0.22)
                theta = np.random.uniform(0.0, 2.0 * math.pi)
                x0 = xx - cx
                y0 = yy - cy
                xr = np.cos(theta) * x0 + np.sin(theta) * y0
                yr = -np.sin(theta) * x0 + np.cos(theta) * y0
                mask = (np.abs(xr) <= wx / 2.0) & (np.abs(yr) <= wy / 2.0)

            field[mask] = 1.0

    if family in ["fronts", "mixed"]:
        theta = np.random.uniform(0.0, 2.0 * math.pi)
        offset = np.random.uniform(-0.2, 0.2)
        ramp = np.cos(theta) * (xx - 0.5) + np.sin(theta) * (yy - 0.5) - offset
        front = np.tanh(25.0 * ramp).astype(np.float32)
        field = np.where(np.abs(front) > 0.4, front, field)

    if family in ["rings", "mixed"]:
        cx = np.random.uniform(0.25, 0.75)
        cy = np.random.uniform(0.25, 0.75)
        r0 = np.random.uniform(0.10, 0.24)
        width = np.random.uniform(0.02, 0.05)
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        ring = np.exp(-((dist - r0) ** 2) / (2 * width**2))
        field[ring > 0.35] = 1.0

    field = field + 0.05 * np.random.randn(nx, ny).astype(np.float32)
    return np.clip(field, -1.0, 1.0).astype(np.float32)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5), dpi=140)

for i in range(4):
    u0 = sample_initial_condition(cfg.nx, cfg.ny)
    uT = solve_allen_cahn_final_state(u0, cfg)

    axes[0, i].imshow(u0, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0, i].set_title("Initial condition", fontsize=10)
    axes[0, i].axis("off")

    axes[1, i].imshow(uT, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[1, i].set_title("Later state", fontsize=10)
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def build_dataset(config: AllenCahn2DConfig):
    n_total = config.n_train + config.n_val + config.n_test

    x_all = []
    y_clean_all = []

    for _ in range(n_total):
        u0 = sample_initial_condition(config.nx, config.ny)
        uT = solve_allen_cahn_final_state(u0, config)

        x_all.append(u0[None, :, :])
        y_clean_all.append(uT[None, :, :])

    x_all = torch.tensor(np.stack(x_all, axis=0), dtype=torch.float32)
    y_clean_all = torch.tensor(np.stack(y_clean_all, axis=0), dtype=torch.float32)
    y_noisy_all = torch.clamp(y_clean_all + config.noise_std * torch.randn_like(y_clean_all), -1.0, 1.0)

    x_train = x_all[:config.n_train]
    x_val = x_all[config.n_train: config.n_train + config.n_val]
    x_test = x_all[-config.n_test:]

    y_clean_train = y_clean_all[:config.n_train]
    y_clean_val = y_clean_all[config.n_train: config.n_train + config.n_val]
    y_clean_test = y_clean_all[-config.n_test:]

    y_noisy_train = y_noisy_all[:config.n_train]
    y_noisy_val = y_noisy_all[config.n_train: config.n_train + config.n_val]
    y_noisy_test = y_noisy_all[-config.n_test:]

    y_mean = y_clean_train.mean(dim=(0, 2, 3), keepdim=True)
    y_std = y_clean_train.std(dim=(0, 2, 3), keepdim=True).clamp_min(1e-6)

    return {
        "x_train": x_train,
        "x_val": x_val,
        "x_test": x_test,
        "y_clean_train": y_clean_train,
        "y_clean_val": y_clean_val,
        "y_clean_test": y_clean_test,
        "y_noisy_train": y_noisy_train,
        "y_noisy_val": y_noisy_val,
        "y_noisy_test": y_noisy_test,
        "y_clean_train_norm": (y_clean_train - y_mean) / y_std,
        "y_clean_val_norm": (y_clean_val - y_mean) / y_std,
        "y_clean_test_norm": (y_clean_test - y_mean) / y_std,
        "y_noisy_train_norm": (y_noisy_train - y_mean) / y_std,
        "y_noisy_val_norm": (y_noisy_val - y_mean) / y_std,
        "y_noisy_test_norm": (y_noisy_test - y_mean) / y_std,
        "y_mean": y_mean,
        "y_std": y_std,
    }

data = build_dataset(cfg)

for key, value in data.items():
    if isinstance(value, torch.Tensor):
        print(key, tuple(value.shape))

In [ ]:
forward_train_loader = DataLoader(
    TensorDataset(data["x_train"], data["y_clean_train_norm"]),
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=False,
)

forward_val_loader = DataLoader(
    TensorDataset(data["x_val"], data["y_clean_val_norm"]),
    batch_size=cfg.batch_size,
    shuffle=False,
    drop_last=False,
)

reverse_train_loader = DataLoader(
    TensorDataset(data["x_train"], data["y_clean_train_norm"]),
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=False,
)

reverse_val_loader = DataLoader(
    TensorDataset(data["x_val"], data["y_clean_val_norm"]),
    batch_size=cfg.batch_size,
    shuffle=False,
    drop_last=False,
)

reverse_jcp_train_loader = DataLoader(
    TensorDataset(data["x_train"], data["y_clean_train_norm"]),
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=False,
)

reverse_jcp_val_loader = DataLoader(
    TensorDataset(data["x_val"], data["y_clean_val_norm"]),
    batch_size=cfg.batch_size,
    shuffle=False,
    drop_last=False,
)

In [ ]:
class ConvResidualBlock(nn.Module):
    def __init__(self, channels: int, negative_slope: float):
        super().__init__()
        self.conv1 = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1, padding_mode="circular"
        )
        self.conv2 = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1, padding_mode="circular"
        )
        self.negative_slope = negative_slope

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = F.leaky_relu(self.conv1(x), negative_slope=self.negative_slope)
        z = self.conv2(z)
        return F.leaky_relu(x + z, negative_slope=self.negative_slope)

In [ ]:
class ForwardCNN(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, hidden_channels: int, num_res_blocks: int, negative_slope: float):
        super().__init__()
        self.negative_slope = negative_slope
        self.entry = nn.Conv2d(
            in_channels, hidden_channels, kernel_size=3, padding=1, padding_mode="circular"
        )
        self.blocks = nn.ModuleList([
            ConvResidualBlock(hidden_channels, negative_slope)
            for _ in range(num_res_blocks)
        ])
        self.exit = nn.Conv2d(
            hidden_channels, out_channels, kernel_size=3, padding=1, padding_mode="circular"
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = F.leaky_relu(self.entry(x), negative_slope=self.negative_slope)
        for block in self.blocks:
            z = block(z)
        return self.exit(z)

In [ ]:
class ReverseCNN(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, num_res_blocks: int, negative_slope: float):
        super().__init__()
        self.negative_slope = negative_slope
        self.entry = nn.Conv2d(
            in_channels, hidden_channels, kernel_size=3, padding=1, padding_mode="circular"
        )
        self.blocks = nn.ModuleList([
            ConvResidualBlock(hidden_channels, negative_slope)
            for _ in range(num_res_blocks)
        ])
        self.exit = nn.Conv2d(
            hidden_channels, 1, kernel_size=3, padding=1, padding_mode="circular"
        )

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        z = F.leaky_relu(self.entry(y), negative_slope=self.negative_slope)
        for block in self.blocks:
            z = block(z)
        return torch.tanh(self.exit(z))

In [ ]:
class DeceptronAllenCahnModel(nn.Module):
    def __init__(self, x_channels: int, y_channels: int, hidden_channels: int, num_res_blocks: int, negative_slope: float):
        super().__init__()
        self.forward_map = ForwardCNN(
            x_channels, y_channels, hidden_channels, num_res_blocks, negative_slope
        )
        self.reverse_map = ReverseCNN(
            y_channels, hidden_channels, num_res_blocks, negative_slope
        )

    def f(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward_map(x)

    def g(self, y: torch.Tensor) -> torch.Tensor:
        return self.reverse_map(y)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.f(x)

In [ ]:
def single_sample_probe_jcp(model: DeceptronAllenCahnModel, x: torch.Tensor, num_probes: int = 1) -> torch.Tensor:
    x_shape = tuple(x.shape)
    x_vec = x.reshape(-1)
    total = x_vec.new_tensor(0.0)

    def f_single(inp: torch.Tensor) -> torch.Tensor:
        inp_reshaped = inp.reshape(1, *x_shape)
        return model.f(inp_reshaped).reshape(-1)

    y0 = f_single(x_vec)

    def g_single(inp: torch.Tensor) -> torch.Tensor:
        inp_reshaped = inp.reshape(1, cfg.y_channels, cfg.nx, cfg.ny)
        return model.g(inp_reshaped).reshape(-1)

    for _ in range(num_probes):
        xi = torch.empty_like(x_vec).bernoulli_(0.5).mul_(2.0).sub_(1.0)
        _, jf_xi = jvp(f_single, (x_vec,), (xi,))
        _, jg_jf_xi = jvp(g_single, (y0,), (jf_xi,))
        total = total + torch.mean((jg_jf_xi - xi) ** 2)

    return total / num_probes

In [ ]:
def batch_probe_jcp(model: DeceptronAllenCahnModel, x_batch: torch.Tensor, num_probes: int = 1) -> torch.Tensor:
    vals = []
    for i in range(x_batch.shape[0]):
        vals.append(single_sample_probe_jcp(model, x_batch[i], num_probes=num_probes))
    return torch.stack(vals).mean()

In [ ]:
@torch.no_grad()
def estimate_rjcp_dataset(model: DeceptronAllenCahnModel, x_data: torch.Tensor, num_probes: int = 2, max_samples: int = 24) -> float:
    model.eval()
    x_subset = x_data[:max_samples].to(device)
    vals = []
    for i in range(x_subset.shape[0]):
        vals.append(single_sample_probe_jcp(model, x_subset[i], num_probes=num_probes).detach().cpu())
    return float(torch.stack(vals).mean().item())

In [ ]:
@torch.no_grad()
def evaluate_forward_validation_loss(model: DeceptronAllenCahnModel, loader) -> float:
    model.eval()
    total = 0.0
    count = 0
    for x_cpu, y_cpu in loader:
        x = x_cpu.to(device)
        y = y_cpu.to(device)
        y_pred = model.f(x)
        total += float(F.mse_loss(y_pred, y, reduction="sum").item())
        count += y.numel()
    return total / max(count, 1)

In [ ]:
@torch.no_grad()
def evaluate_reverse_validation(model: DeceptronAllenCahnModel, loader) -> Dict[str, float]:
    model.eval()
    totals = {"reconstruction": 0.0, "cycle": 0.0, "loss": 0.0}
    count = 0

    for x_cpu, y_cpu in loader:
        x = x_cpu.to(device)
        y = y_cpu.to(device)

        x_hat = model.g(y)
        y_cyc = model.f(x_hat)

        rec_loss = F.mse_loss(x_hat, x)
        cyc_loss = F.mse_loss(y_cyc, y)
        loss = cfg.reconstruction_weight * rec_loss + cfg.cycle_weight * cyc_loss

        bs = x.shape[0]
        totals["reconstruction"] += float(rec_loss.item()) * bs
        totals["cycle"] += float(cyc_loss.item()) * bs
        totals["loss"] += float(loss.item()) * bs
        count += bs

    return {k: v / count for k, v in totals.items()}

In [ ]:
def freeze_forward_map(model: DeceptronAllenCahnModel):
    for p in model.forward_map.parameters():
        p.requires_grad = False

In [ ]:
def train_forward_map_only(config: AllenCahn2DConfig, train_loader, val_loader):
    model = DeceptronAllenCahnModel(
        config.x_channels,
        config.y_channels,
        config.hidden_channels,
        config.num_res_blocks,
        config.negative_slope,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.forward_map.parameters(),
        lr=config.forward_lr,
        weight_decay=config.forward_weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.forward_epochs)

    history = []
    best_state = None
    best_val = float("inf")

    for epoch in range(config.forward_epochs):
        model.train()
        train_total = 0.0
        count = 0

        for x_cpu, y_cpu in train_loader:
            x = x_cpu.to(device)
            y = y_cpu.to(device)

            optimizer.zero_grad(set_to_none=True)
            loss = F.mse_loss(model.f(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.forward_map.parameters(), config.gradient_clip)
            optimizer.step()

            train_total += float(loss.item()) * x.shape[0]
            count += x.shape[0]

        scheduler.step()

        val_task = evaluate_forward_validation_loss(model, val_loader)
        val_rjcp = estimate_rjcp_dataset(model, data["x_val"], config.jcp_num_probes_eval, config.eval_subset_rjcp)

        row = {
            "epoch": epoch + 1,
            "train_task": train_total / count,
            "val_task": val_task,
            "val_rjcp": val_rjcp,
        }
        history.append(row)

        if val_task < best_val:
            best_val = val_task
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Forward epoch {epoch+1:03d} | train_task={row['train_task']:.6f} | val_task={row['val_task']:.6f} | val_rjcp={row['val_rjcp']:.6f}")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
def train_reverse_map_only(config: AllenCahn2DConfig, model_init: DeceptronAllenCahnModel, train_loader, val_loader):
    model = DeceptronAllenCahnModel(
        config.x_channels,
        config.y_channels,
        config.hidden_channels,
        config.num_res_blocks,
        config.negative_slope,
    ).to(device)

    model.forward_map.load_state_dict({k: v.detach().clone() for k, v in model_init.forward_map.state_dict().items()})
    freeze_forward_map(model)

    optimizer = torch.optim.Adam(
        model.reverse_map.parameters(),
        lr=config.reverse_lr,
        weight_decay=config.reverse_weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.reverse_epochs)

    history = []
    best_state = None
    best_score = float("inf")

    for epoch in range(config.reverse_epochs):
        model.train()
        totals = {"loss": 0.0, "reconstruction": 0.0, "cycle": 0.0}
        count = 0

        for x_cpu, y_cpu in train_loader:
            x = x_cpu.to(device)
            y = y_cpu.to(device)

            optimizer.zero_grad(set_to_none=True)

            x_hat = model.g(y)
            y_tilde = y + config.y_tilde_noise * torch.randn_like(y)
            y_cyc = model.f(model.g(y_tilde))

            rec_loss = F.mse_loss(x_hat, x)
            cyc_loss = F.mse_loss(y_cyc, y_tilde)
            loss = config.reconstruction_weight * rec_loss + config.cycle_weight * cyc_loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.reverse_map.parameters(), config.gradient_clip)
            optimizer.step()

            bs = x.shape[0]
            count += bs
            totals["loss"] += float(loss.item()) * bs
            totals["reconstruction"] += float(rec_loss.item()) * bs
            totals["cycle"] += float(cyc_loss.item()) * bs

        scheduler.step()

        val_stats = evaluate_reverse_validation(model, val_loader)
        val_rjcp = estimate_rjcp_dataset(model, data["x_val"], config.jcp_num_probes_eval, config.eval_subset_rjcp)

        row = {
            "epoch": epoch + 1,
            "train_loss": totals["loss"] / count,
            "train_reconstruction": totals["reconstruction"] / count,
            "train_cycle": totals["cycle"] / count,
            "val_loss": val_stats["loss"],
            "val_reconstruction": val_stats["reconstruction"],
            "val_cycle": val_stats["cycle"],
            "val_rjcp": val_rjcp,
        }
        history.append(row)

        score = row["val_reconstruction"] + 0.05 * row["val_rjcp"]
        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Reverse epoch {epoch+1:03d} | val_reconstruction={row['val_reconstruction']:.6f} | val_rjcp={row['val_rjcp']:.6f}")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
def train_reverse_map_with_jcp_fixed_forward(
    config: AllenCahn2DConfig,
    model_init: DeceptronAllenCahnModel,
    train_loader,
    val_loader,
    use_jcp: bool,
    reference_reconstruction: float,
):
    model = DeceptronAllenCahnModel(
        config.x_channels,
        config.y_channels,
        config.hidden_channels,
        config.num_res_blocks,
        config.negative_slope,
    ).to(device)

    model.forward_map.load_state_dict({k: v.detach().clone() for k, v in model_init.forward_map.state_dict().items()})
    model.reverse_map.load_state_dict({k: v.detach().clone() for k, v in model_init.reverse_map.state_dict().items()})
    freeze_forward_map(model)

    optimizer = torch.optim.Adam(
        model.reverse_map.parameters(),
        lr=config.reverse_jcp_lr,
        weight_decay=config.reverse_jcp_weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.reverse_jcp_epochs)

    history = []
    best_state = None
    best_score = float("inf")

    for epoch in range(config.reverse_jcp_epochs):
        model.train()
        totals = {"loss": 0.0, "reconstruction": 0.0, "cycle": 0.0, "probe_jcp": 0.0}
        count = 0

        for x_cpu, y_cpu in train_loader:
            x = x_cpu.to(device)
            y = y_cpu.to(device)

            optimizer.zero_grad(set_to_none=True)

            x_hat = model.g(y)
            y_tilde = y + config.y_tilde_noise * torch.randn_like(y)
            y_cyc = model.f(model.g(y_tilde))

            rec_loss = F.mse_loss(x_hat, x)
            cyc_loss = F.mse_loss(y_cyc, y_tilde)

            if use_jcp:
                bs_jcp = min(config.jcp_batch_subsample, x.shape[0])
                y_subset = y[:bs_jcp]
                with torch.no_grad():
                    x_pred_subset = model.g(y_subset)

                probe_jcp_loss = config.probe_jcp_weight * batch_probe_jcp(
                    model,
                    x_pred_subset,
                    num_probes=config.jcp_num_probes_train,
                )
            else:
                probe_jcp_loss = torch.tensor(0.0, device=device)

            loss = config.reconstruction_weight * rec_loss + config.cycle_weight * cyc_loss + probe_jcp_loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.reverse_map.parameters(), config.gradient_clip)
            optimizer.step()

            bs = x.shape[0]
            count += bs
            totals["loss"] += float(loss.item()) * bs
            totals["reconstruction"] += float(rec_loss.item()) * bs
            totals["cycle"] += float(cyc_loss.item()) * bs
            totals["probe_jcp"] += float(probe_jcp_loss.item()) * bs

        scheduler.step()

        val_stats = evaluate_reverse_validation(model, val_loader)
        val_rjcp = estimate_rjcp_dataset(model, data["x_val"], config.jcp_num_probes_eval, config.eval_subset_rjcp)

        row = {
            "epoch": epoch + 1,
            "train_loss": totals["loss"] / count,
            "train_reconstruction": totals["reconstruction"] / count,
            "train_cycle": totals["cycle"] / count,
            "train_probe_jcp": totals["probe_jcp"] / count,
            "val_loss": val_stats["loss"],
            "val_reconstruction": val_stats["reconstruction"],
            "val_cycle": val_stats["cycle"],
            "val_rjcp": val_rjcp,
        }
        history.append(row)

        recon_ratio = row["val_reconstruction"] / max(reference_reconstruction, 1e-12)
        if recon_ratio <= config.recon_guard_factor:
            score = row["val_rjcp"] + 1e-3 * row["val_loss"]
        else:
            score = row["val_rjcp"] + 1000.0 * (recon_ratio - config.recon_guard_factor)

        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Reverse+JCP epoch {epoch+1:03d} | val_reconstruction={row['val_reconstruction']:.6f} | val_rjcp={row['val_rjcp']:.6f} | probe_jcp={row['train_probe_jcp']:.6f}")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
model_stage1, history_forward = train_forward_map_only(cfg, forward_train_loader, forward_val_loader)
print("best stage-1 val_task:", history_forward["val_task"].min())

model_stage2, history_reverse = train_reverse_map_only(cfg, model_stage1, reverse_train_loader, reverse_val_loader)
print("best stage-2 val_reconstruction:", history_reverse["val_reconstruction"].min())
print("best stage-2 val_rjcp:", history_reverse["val_rjcp"].min())

reference_recon = float(history_reverse["val_reconstruction"].min())

model_plus_jcp, history_reverse_jcp_plus = train_reverse_map_with_jcp_fixed_forward(
    cfg, model_stage2, reverse_jcp_train_loader, reverse_jcp_val_loader, True, reference_recon
)

model_minus_jcp, history_reverse_jcp_minus = train_reverse_map_with_jcp_fixed_forward(
    cfg, model_stage2, reverse_jcp_train_loader, reverse_jcp_val_loader, False, reference_recon
)

val_stats_plus = evaluate_reverse_validation(model_plus_jcp, reverse_jcp_val_loader)
val_stats_minus = evaluate_reverse_validation(model_minus_jcp, reverse_jcp_val_loader)

print("+JCP val_reconstruction:", val_stats_plus["reconstruction"])
print("-JCP val_reconstruction:", val_stats_minus["reconstruction"])
print("+JCP val RJCP:", estimate_rjcp_dataset(model_plus_jcp, data["x_val"], cfg.jcp_num_probes_eval, cfg.eval_subset_rjcp))
print("-JCP val RJCP:", estimate_rjcp_dataset(model_minus_jcp, data["x_val"], cfg.jcp_num_probes_eval, cfg.eval_subset_rjcp))

In [ ]:
def rmse_x(x_hat: torch.Tensor, x_true: torch.Tensor) -> float:
    return torch.sqrt(torch.mean((x_hat - x_true) ** 2)).item()

def phi_value(forward_model: nn.Module, x: torch.Tensor, y_target: torch.Tensor) -> torch.Tensor:
    return 0.5 * torch.mean((forward_model(x) - y_target) ** 2)

def phi_grad(forward_model: nn.Module, x: torch.Tensor, y_target: torch.Tensor):
    x_req = x.detach().clone().requires_grad_(True)
    residual = forward_model(x_req) - y_target
    phi = 0.5 * torch.mean(residual * residual)
    grad = torch.autograd.grad(phi, x_req)[0]
    return phi.detach(), grad.detach(), residual.detach()

def normalized_residual(forward_model: nn.Module, x: torch.Tensor, y_target: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean((forward_model(x) - y_target) ** 2))

def clamp_box(x: torch.Tensor) -> torch.Tensor:
    return torch.clamp(x, -1.0, 1.0)

In [ ]:
def armijo_accept(forward_model: nn.Module, x_t: torch.Tensor, p: torch.Tensor, y_target: torch.Tensor, rho: float, c: float):
    phi_t, grad_t, _ = phi_grad(forward_model, x_t, y_target)
    x_trial = clamp_box((1.0 - rho) * x_t + rho * (x_t + p))
    phi_trial = phi_value(forward_model, x_trial, y_target)
    rhs = phi_t + c * rho * torch.sum(grad_t * p)
    return x_trial, phi_trial, rhs

In [ ]:
def jvp_forward(forward_model: nn.Module, x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    def f_single(inp):
        return forward_model(inp)
    _, jv = jvp(f_single, (x,), (v,))
    return jv

def vjp_forward(forward_model: nn.Module, x: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    x_req = x.detach().clone().requires_grad_(True)
    y = forward_model(x_req)
    scalar = torch.sum(y * u)
    return torch.autograd.grad(scalar, x_req)[0]

def normal_matvec(forward_model: nn.Module, x: torch.Tensor, v: torch.Tensor, damping: float = 0.0) -> torch.Tensor:
    jv = jvp_forward(forward_model, x, v)
    jt_j_v = vjp_forward(forward_model, x, jv)
    return jt_j_v / float(jv.numel()) + damping * v

def conjugate_gradient(matvec, b, max_iter=30, tol=1e-6):
    x = torch.zeros_like(b)
    r = b - matvec(x)
    p = r.clone()
    rs_old = torch.sum(r * r)

    for _ in range(max_iter):
        Ap = matvec(p)
        alpha = rs_old / torch.sum(p * Ap).clamp_min(1e-12)
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = torch.sum(r * r)
        if torch.sqrt(rs_new).item() < tol:
            break
        p = r + (rs_new / rs_old.clamp_min(1e-12)) * p
        rs_old = rs_new

    return x

In [ ]:
def solve_dipg(model: DeceptronAllenCahnModel, y_target: torch.Tensor, x0: torch.Tensor, config: AllenCahn2DConfig, num_probes_eval: int = 2) -> Dict:
    model.eval()
    x = x0.clone().to(device)

    accepted_steps = 0
    alpha_accept = []
    step_norm_accept = []
    cosine_accept = []
    residual_drop_accept = []

    t0 = time.perf_counter()

    for iteration in range(config.max_iterations):
        _, grad_t, residual_t = phi_grad(model.f, x, y_target)
        residual_before = torch.sqrt(torch.mean(residual_t * residual_t)).item()

        if residual_before <= config.tolerance_eps:
            break

        grad_flat = grad_t.reshape(-1)
        grad_norm = torch.norm(grad_flat).item()

        alpha = config.dipg_step_size
        accepted = False
        y_t = model.f(x)

        for _ in range(config.max_backtracking_steps + 1):
            y_proposed = y_t - alpha * residual_t
            x_proposed = model.g(y_proposed)
            p = x_proposed - x

            x_trial, phi_trial, rhs = armijo_accept(model.f, x, p, y_target, config.rho, config.armijo_c)

            if phi_trial <= rhs:
                step_flat = p.reshape(-1)
                step_norm = torch.norm(step_flat).item()

                cos_val = np.nan
                if grad_norm > 1e-12 and step_norm > 1e-12:
                    cos_val = torch.dot(step_flat, (-grad_flat)) / (torch.norm(step_flat) * torch.norm(grad_flat))
                    cos_val = float(cos_val.item())

                x = x_trial.detach()
                residual_after = normalized_residual(model.f, x, y_target).item()

                accepted = True
                accepted_steps += 1
                alpha_accept.append(alpha)
                step_norm_accept.append(step_norm)
                cosine_accept.append(cos_val)
                residual_drop_accept.append(residual_before - residual_after)
                break

            alpha *= 0.5

        if not accepted:
            break

    elapsed = time.perf_counter() - t0
    final_residual = normalized_residual(model.f, x, y_target).item()
    final_rjcp = float(single_sample_probe_jcp(model, x.detach().squeeze(0), num_probes=num_probes_eval).detach().item())

    return {
        "x_hat": x.detach().cpu(),
        "iters": iteration + 1,
        "success": float(False),
        "final_residual": final_residual,
        "accepted_frac": accepted_steps / max(iteration + 1, 1),
        "time_sec": elapsed,
        "final_rjcp": final_rjcp,
        "residual_drop_per_accept": float(np.mean(residual_drop_accept)) if len(residual_drop_accept) else np.nan,
        "alpha_accept": float(np.mean(alpha_accept)) if len(alpha_accept) else np.nan,
        "step_norm_accept": float(np.mean(step_norm_accept)) if len(step_norm_accept) else np.nan,
        "cosine_with_neg_grad": float(np.mean(cosine_accept)) if len(cosine_accept) else np.nan,
    }

In [ ]:
def solve_gradient_descent(forward_model: nn.Module, y_target: torch.Tensor, x0: torch.Tensor, config: AllenCahn2DConfig) -> Dict:
    x = x0.clone().to(device)
    accepted_steps = 0
    t0 = time.perf_counter()

    for iteration in range(config.max_iterations):
        _, grad_t, residual_t = phi_grad(forward_model, x, y_target)
        residual_before = torch.sqrt(torch.mean(residual_t * residual_t)).item()
        if residual_before <= config.tolerance_eps:
            break

        alpha = config.gd_step_size
        accepted = False

        for _ in range(config.max_backtracking_steps + 1):
            p = -alpha * grad_t
            x_trial, phi_trial, rhs = armijo_accept(forward_model, x, p, y_target, config.rho, config.armijo_c)
            if phi_trial <= rhs:
                x = x_trial.detach()
                accepted = True
                accepted_steps += 1
                break
            alpha *= 0.5

        if not accepted:
            break

    elapsed = time.perf_counter() - t0
    return {
        "x_hat": x.detach().cpu(),
        "iters": iteration + 1,
        "success": float(False),
        "final_residual": normalized_residual(forward_model, x, y_target).item(),
        "accepted_frac": accepted_steps / max(iteration + 1, 1),
        "time_sec": elapsed,
        "final_rjcp": np.nan,
        "residual_drop_per_accept": np.nan,
        "alpha_accept": np.nan,
        "step_norm_accept": np.nan,
        "cosine_with_neg_grad": np.nan,
    }

In [ ]:
def solve_gauss_newton(forward_model: nn.Module, y_target: torch.Tensor, x0: torch.Tensor, config: AllenCahn2DConfig) -> Dict:
    x = x0.clone().to(device)
    accepted_steps = 0
    t0 = time.perf_counter()

    for iteration in range(config.max_iterations):
        _, _, residual_t = phi_grad(forward_model, x, y_target)
        residual_value = torch.sqrt(torch.mean(residual_t * residual_t)).item()
        if residual_value <= config.tolerance_eps:
            break

        rhs = -vjp_forward(forward_model, x, residual_t) / float(residual_t.numel())

        def A_fn(v):
            return normal_matvec(forward_model, x, v, damping=0.0)

        dx = conjugate_gradient(A_fn, rhs, max_iter=config.cg_max_iter, tol=config.cg_tol)

        alpha = config.gn_step_size
        accepted = False
        for _ in range(config.max_backtracking_steps + 1):
            p = alpha * dx
            x_trial, phi_trial, rhs_armijo = armijo_accept(forward_model, x, p, y_target, config.rho, config.armijo_c)
            if phi_trial <= rhs_armijo:
                x = x_trial.detach()
                accepted = True
                accepted_steps += 1
                break
            alpha *= 0.5

        if not accepted:
            break

    elapsed = time.perf_counter() - t0
    return {
        "x_hat": x.detach().cpu(),
        "iters": iteration + 1,
        "success": float(False),
        "final_residual": normalized_residual(forward_model, x, y_target).item(),
        "accepted_frac": accepted_steps / max(iteration + 1, 1),
        "time_sec": elapsed,
        "final_rjcp": np.nan,
        "residual_drop_per_accept": np.nan,
        "alpha_accept": np.nan,
        "step_norm_accept": np.nan,
        "cosine_with_neg_grad": np.nan,
    }

In [ ]:
def solve_lm(forward_model: nn.Module, y_target: torch.Tensor, x0: torch.Tensor, config: AllenCahn2DConfig) -> Dict:
    x = x0.clone().to(device)
    accepted_steps = 0
    t0 = time.perf_counter()

    for iteration in range(config.max_iterations):
        _, _, residual_t = phi_grad(forward_model, x, y_target)
        residual_value = torch.sqrt(torch.mean(residual_t * residual_t)).item()
        if residual_value <= config.tolerance_eps:
            break

        rhs = -vjp_forward(forward_model, x, residual_t) / float(residual_t.numel())

        def A_fn(v):
            return normal_matvec(forward_model, x, v, damping=config.lm_damping)

        dx = conjugate_gradient(A_fn, rhs, max_iter=config.cg_max_iter, tol=config.cg_tol)

        alpha = config.lm_step_size
        accepted = False
        for _ in range(config.max_backtracking_steps + 1):
            p = alpha * dx
            x_trial, phi_trial, rhs_armijo = armijo_accept(forward_model, x, p, y_target, config.rho, config.armijo_c)
            if phi_trial <= rhs_armijo:
                x = x_trial.detach()
                accepted = True
                accepted_steps += 1
                break
            alpha *= 0.5

        if not accepted:
            break

    elapsed = time.perf_counter() - t0
    return {
        "x_hat": x.detach().cpu(),
        "iters": iteration + 1,
        "success": float(False),
        "final_residual": normalized_residual(forward_model, x, y_target).item(),
        "accepted_frac": accepted_steps / max(iteration + 1, 1),
        "time_sec": elapsed,
        "final_rjcp": np.nan,
        "residual_drop_per_accept": np.nan,
        "alpha_accept": np.nan,
        "step_norm_accept": np.nan,
        "cosine_with_neg_grad": np.nan,
    }

In [ ]:
def solve_lbfgs(forward_model: nn.Module, y_target: torch.Tensor, x0: torch.Tensor, config: AllenCahn2DConfig) -> Dict:
    x = nn.Parameter(x0.clone().to(device).contiguous())
    y_target = y_target.to(device).contiguous()

    optimizer = torch.optim.LBFGS(
        [x],
        lr=config.lbfgs_lr,
        max_iter=config.lbfgs_max_iter,
        history_size=config.lbfgs_history_size,
        tolerance_grad=config.lbfgs_tolerance_grad,
        tolerance_change=config.lbfgs_tolerance_change,
        line_search_fn="strong_wolfe",
    )

    iteration_counter = {"n": 0}
    t0 = time.perf_counter()

    def closure():
        optimizer.zero_grad(set_to_none=True)
        x_clamped = torch.clamp(x, -1.0, 1.0).contiguous()
        loss = 0.5 * torch.mean((forward_model(x_clamped) - y_target) ** 2)
        loss.backward()
        if x.grad is not None:
            x.grad = x.grad.contiguous()
        iteration_counter["n"] += 1
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        x_final = torch.clamp(x.detach(), -1.0, 1.0).contiguous()

    elapsed = time.perf_counter() - t0
    return {
        "x_hat": x_final.cpu(),
        "iters": iteration_counter["n"],
        "success": float(False),
        "final_residual": normalized_residual(forward_model, x_final, y_target).item(),
        "accepted_frac": 1.0,
        "time_sec": elapsed,
        "final_rjcp": np.nan,
        "residual_drop_per_accept": np.nan,
        "alpha_accept": np.nan,
        "step_norm_accept": np.nan,
        "cosine_with_neg_grad": np.nan,
    }

In [ ]:
def evaluate_allen_cahn_methods(case_x: torch.Tensor, case_y: torch.Tensor, model_plus: DeceptronAllenCahnModel, model_minus: DeceptronAllenCahnModel, config: AllenCahn2DConfig) -> pd.DataFrame:
    rows = []

    method_specs = [
        ("+JCP", "D-IPG", lambda y_star, x0: solve_dipg(model_plus, y_star, x0, config, num_probes_eval=2)),
        ("-JCP", "D-IPG", lambda y_star, x0: solve_dipg(model_minus, y_star, x0, config, num_probes_eval=2)),
        ("Baseline", "Gradient Descent", lambda y_star, x0: solve_gradient_descent(model_plus.f, y_star, x0, config)),
        ("Baseline", "GN", lambda y_star, x0: solve_gauss_newton(model_plus.f, y_star, x0, config)),
        ("Baseline", "LM", lambda y_star, x0: solve_lm(model_plus.f, y_star, x0, config)),
        ("Baseline", "L-BFGS", lambda y_star, x0: solve_lbfgs(model_plus.f, y_star, x0, config)),
    ]

    for ablation, method_name, method_function in method_specs:
        per_instance = []

        for i in range(case_x.shape[0]):
            x_true = case_x[i:i+1]
            y_star = case_y[i:i+1].to(device)
            x0 = torch.zeros_like(x_true).to(device)

            result = method_function(y_star, x0)
            rmse_val = rmse_x(result["x_hat"], x_true)
            result["success"] = float(rmse_val <= config.success_rmse_threshold)

            per_instance.append({
                "rmse_x": rmse_val,
                **{k: v for k, v in result.items() if k != "x_hat"},
            })

        df = pd.DataFrame(per_instance)
        summary = {
            "ablation": ablation,
            "method": method_name,
            "num_instances": len(df),
            "success_rate": df["success"].mean(),
            "mean_iters": df["iters"].mean(),
            "median_iters": df["iters"].median(),
            "mean_rmse_x": df["rmse_x"].mean(),
            "mean_final_residual": df["final_residual"].mean(),
            "mean_accepted_frac": df["accepted_frac"].mean(),
            "mean_time_sec": df["time_sec"].mean(),
            "mean_final_rjcp": df["final_rjcp"].mean(),
            "mean_residual_drop_per_accept": df["residual_drop_per_accept"].mean(),
            "mean_alpha_accept": df["alpha_accept"].mean(),
            "mean_step_norm_accept": df["step_norm_accept"].mean(),
            "mean_cosine_with_neg_grad": df["cosine_with_neg_grad"].mean(),
        }
        rows.append(summary)
        print("done:", ablation, method_name)

    return pd.DataFrame(rows)

In [ ]:
main_results = evaluate_allen_cahn_methods(
    data["x_test"],
    data["y_noisy_test_norm"],
    model_plus_jcp,
    model_minus_jcp,
    cfg,
)

print(main_results.round(6).to_string(index=False))

In [ ]:
rjcp_summary = pd.DataFrame([
    {
        "ablation": "+JCP",
        "val_reconstruction": evaluate_reverse_validation(model_plus_jcp, reverse_jcp_val_loader)["reconstruction"],
        "val_rjcp": estimate_rjcp_dataset(model_plus_jcp, data["x_val"], cfg.jcp_num_probes_eval, cfg.eval_subset_rjcp),
        "test_rjcp": estimate_rjcp_dataset(model_plus_jcp, data["x_test"], cfg.jcp_num_probes_eval, cfg.eval_subset_rjcp),
    },
    {
        "ablation": "-JCP",
        "val_reconstruction": evaluate_reverse_validation(model_minus_jcp, reverse_jcp_val_loader)["reconstruction"],
        "val_rjcp": estimate_rjcp_dataset(model_minus_jcp, data["x_val"], cfg.jcp_num_probes_eval, cfg.eval_subset_rjcp),
        "test_rjcp": estimate_rjcp_dataset(model_minus_jcp, data["x_test"], cfg.jcp_num_probes_eval, cfg.eval_subset_rjcp),
    },
])

print(rjcp_summary.round(6).to_string(index=False))

In [ ]:
history_forward.to_csv("/content/allen_cahn2d_stage1_forward_history.csv", index=False)
history_reverse.to_csv("/content/allen_cahn2d_stage2_reverse_history.csv", index=False)
history_reverse_jcp_plus.to_csv("/content/allen_cahn2d_stage3_reverse_plus_jcp_history.csv", index=False)
history_reverse_jcp_minus.to_csv("/content/allen_cahn2d_stage3_reverse_minus_jcp_history.csv", index=False)

main_results.to_csv("/content/allen_cahn2d_main_results.csv", index=False)
rjcp_summary.to_csv("/content/allen_cahn2d_rjcp_summary.csv", index=False)

torch.save(model_stage1.state_dict(), "/content/allen_cahn2d_model_stage1.pt")
torch.save(model_stage2.state_dict(), "/content/allen_cahn2d_model_stage2.pt")
torch.save(model_plus_jcp.state_dict(), "/content/allen_cahn2d_model_plus_jcp.pt")
torch.save(model_minus_jcp.state_dict(), "/content/allen_cahn2d_model_minus_jcp.pt")

##Plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

def solve_dipg_trace(model, y_target, x0, config, x_true=None, num_probes_eval=2):
    model.eval()
    x = x0.clone().to(device)

    accepted_steps = 0
    rmse_trace = []
    residual_trace = []
    alpha_accept_trace = []
    step_norm_accept_trace = []
    cosine_accept_trace = []
    residual_drop_accept_trace = []

    if x_true is not None:
        rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
    residual_trace.append(normalized_residual(model.f, x, y_target).item())

    for iteration in range(config.max_iterations):
        _, grad_t, residual_t = phi_grad(model.f, x, y_target)
        residual_before = torch.sqrt(torch.mean(residual_t * residual_t)).item()

        if residual_before <= config.tolerance_eps:
            break

        grad_flat = grad_t.reshape(-1)
        grad_norm = torch.norm(grad_flat).item()

        alpha = config.dipg_step_size
        accepted = False
        y_t = model.f(x)

        for _ in range(config.max_backtracking_steps + 1):
            y_proposed = y_t - alpha * residual_t
            x_proposed = model.g(y_proposed)
            p = x_proposed - x

            x_trial, phi_trial, rhs = armijo_accept(
                model.f, x, p, y_target, config.rho, config.armijo_c
            )

            if phi_trial <= rhs:
                step_flat = p.reshape(-1)
                step_norm = torch.norm(step_flat).item()

                cos_val = np.nan
                if grad_norm > 1e-12 and step_norm > 1e-12:
                    cos_val = torch.dot(step_flat, (-grad_flat)) / (torch.norm(step_flat) * torch.norm(grad_flat))
                    cos_val = float(cos_val.item())

                x = x_trial.detach()
                residual_after = normalized_residual(model.f, x, y_target).item()

                accepted = True
                accepted_steps += 1
                alpha_accept_trace.append(alpha)
                step_norm_accept_trace.append(step_norm)
                cosine_accept_trace.append(cos_val)
                residual_drop_accept_trace.append(residual_before - residual_after)

                if x_true is not None:
                    rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
                residual_trace.append(residual_after)
                break

            alpha *= 0.5

        if not accepted:
            break

    final_rjcp = float(
        single_sample_probe_jcp(model, x.detach().squeeze(0), num_probes=num_probes_eval).detach().item()
    )

    return {
        "x_hat": x.detach().cpu(),
        "iters": len(residual_trace) - 1,
        "final_residual": residual_trace[-1],
        "accepted_frac": accepted_steps / max(len(residual_trace) - 1, 1),
        "final_rjcp": final_rjcp,
        "rmse_trace": rmse_trace,
        "residual_trace": residual_trace,
        "alpha_accept_trace": alpha_accept_trace,
        "step_norm_accept_trace": step_norm_accept_trace,
        "cosine_accept_trace": cosine_accept_trace,
        "residual_drop_accept_trace": residual_drop_accept_trace,
    }

def solve_gd_trace(forward_model, y_target, x0, config, x_true=None):
    x = x0.clone().to(device)
    rmse_trace = []
    residual_trace = []

    if x_true is not None:
        rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
    residual_trace.append(normalized_residual(forward_model, x, y_target).item())

    accepted_steps = 0

    for iteration in range(config.max_iterations):
        _, grad_t, residual_t = phi_grad(forward_model, x, y_target)
        residual_before = torch.sqrt(torch.mean(residual_t * residual_t)).item()
        if residual_before <= config.tolerance_eps:
            break

        alpha = config.gd_step_size
        accepted = False

        for _ in range(config.max_backtracking_steps + 1):
            p = -alpha * grad_t
            x_trial, phi_trial, rhs = armijo_accept(
                forward_model, x, p, y_target, config.rho, config.armijo_c
            )
            if phi_trial <= rhs:
                x = x_trial.detach()
                accepted = True
                accepted_steps += 1
                if x_true is not None:
                    rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
                residual_trace.append(normalized_residual(forward_model, x, y_target).item())
                break
            alpha *= 0.5

        if not accepted:
            break

    return {
        "x_hat": x.detach().cpu(),
        "iters": len(residual_trace) - 1,
        "final_residual": residual_trace[-1],
        "accepted_frac": accepted_steps / max(len(residual_trace) - 1, 1),
        "rmse_trace": rmse_trace,
        "residual_trace": residual_trace,
    }

def solve_gn_trace(forward_model, y_target, x0, config, x_true=None):
    x = x0.clone().to(device)
    rmse_trace = []
    residual_trace = []

    if x_true is not None:
        rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
    residual_trace.append(normalized_residual(forward_model, x, y_target).item())

    accepted_steps = 0

    for iteration in range(config.max_iterations):
        _, _, residual_t = phi_grad(forward_model, x, y_target)
        residual_before = torch.sqrt(torch.mean(residual_t * residual_t)).item()
        if residual_before <= config.tolerance_eps:
            break

        rhs = -vjp_forward(forward_model, x, residual_t) / float(residual_t.numel())

        def A_fn(v):
            return normal_matvec(forward_model, x, v, damping=0.0)

        dx = conjugate_gradient(A_fn, rhs, max_iter=config.cg_max_iter, tol=config.cg_tol)

        alpha = config.gn_step_size
        accepted = False

        for _ in range(config.max_backtracking_steps + 1):
            p = alpha * dx
            x_trial, phi_trial, rhs_armijo = armijo_accept(
                forward_model, x, p, y_target, config.rho, config.armijo_c
            )
            if phi_trial <= rhs_armijo:
                x = x_trial.detach()
                accepted = True
                accepted_steps += 1
                if x_true is not None:
                    rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
                residual_trace.append(normalized_residual(forward_model, x, y_target).item())
                break
            alpha *= 0.5

        if not accepted:
            break

    return {
        "x_hat": x.detach().cpu(),
        "iters": len(residual_trace) - 1,
        "final_residual": residual_trace[-1],
        "accepted_frac": accepted_steps / max(len(residual_trace) - 1, 1),
        "rmse_trace": rmse_trace,
        "residual_trace": residual_trace,
    }

def solve_lm_trace(forward_model, y_target, x0, config, x_true=None):
    x = x0.clone().to(device)
    rmse_trace = []
    residual_trace = []

    if x_true is not None:
        rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
    residual_trace.append(normalized_residual(forward_model, x, y_target).item())

    accepted_steps = 0

    for iteration in range(config.max_iterations):
        _, _, residual_t = phi_grad(forward_model, x, y_target)
        residual_before = torch.sqrt(torch.mean(residual_t * residual_t)).item()
        if residual_before <= config.tolerance_eps:
            break

        rhs = -vjp_forward(forward_model, x, residual_t) / float(residual_t.numel())

        def A_fn(v):
            return normal_matvec(forward_model, x, v, damping=config.lm_damping)

        dx = conjugate_gradient(A_fn, rhs, max_iter=config.cg_max_iter, tol=config.cg_tol)

        alpha = config.lm_step_size
        accepted = False

        for _ in range(config.max_backtracking_steps + 1):
            p = alpha * dx
            x_trial, phi_trial, rhs_armijo = armijo_accept(
                forward_model, x, p, y_target, config.rho, config.armijo_c
            )
            if phi_trial <= rhs_armijo:
                x = x_trial.detach()
                accepted = True
                accepted_steps += 1
                if x_true is not None:
                    rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
                residual_trace.append(normalized_residual(forward_model, x, y_target).item())
                break
            alpha *= 0.5

        if not accepted:
            break

    return {
        "x_hat": x.detach().cpu(),
        "iters": len(residual_trace) - 1,
        "final_residual": residual_trace[-1],
        "accepted_frac": accepted_steps / max(len(residual_trace) - 1, 1),
        "rmse_trace": rmse_trace,
        "residual_trace": residual_trace,
    }

def solve_lbfgs_trace(forward_model, y_target, x0, config, x_true=None):
    x = nn.Parameter(x0.clone().to(device).contiguous())
    y_target = y_target.to(device).contiguous()

    rmse_trace = []
    residual_trace = []

    if x_true is not None:
        rmse_trace.append(rmse_x(x.detach().cpu(), x_true.detach().cpu()))
    residual_trace.append(normalized_residual(forward_model, x.detach(), y_target).item())

    optimizer = torch.optim.LBFGS(
        [x],
        lr=config.lbfgs_lr,
        max_iter=config.lbfgs_max_iter,
        history_size=config.lbfgs_history_size,
        tolerance_grad=config.lbfgs_tolerance_grad,
        tolerance_change=config.lbfgs_tolerance_change,
        line_search_fn="strong_wolfe",
    )

    iteration_counter = {"n": 0}

    def closure():
        optimizer.zero_grad(set_to_none=True)
        x_clamped = torch.clamp(x, -1.0, 1.0).contiguous()
        loss = 0.5 * torch.mean((forward_model(x_clamped) - y_target) ** 2)
        loss.backward()
        if x.grad is not None:
            x.grad = x.grad.contiguous()
        iteration_counter["n"] += 1

        with torch.no_grad():
            x_now = torch.clamp(x.detach(), -1.0, 1.0).contiguous()
            if x_true is not None:
                rmse_trace.append(rmse_x(x_now.cpu(), x_true.detach().cpu()))
            residual_trace.append(normalized_residual(forward_model, x_now, y_target).item())

        return loss

    optimizer.step(closure)

    with torch.no_grad():
        x_final = torch.clamp(x.detach(), -1.0, 1.0).contiguous()

    return {
        "x_hat": x_final.cpu(),
        "iters": iteration_counter["n"],
        "final_residual": normalized_residual(forward_model, x_final, y_target).item(),
        "accepted_frac": 1.0,
        "rmse_trace": rmse_trace,
        "residual_trace": residual_trace,
    }

In [ ]:
def pick_representative_cases(case_x, case_y, model_plus, model_minus, cfg, num_cases=3):
    rows = []

    for i in range(case_x.shape[0]):
        x_true = case_x[i:i+1]
        y_star = case_y[i:i+1].to(device)
        x0 = torch.zeros_like(x_true).to(device)

        out_plus = solve_dipg(model_plus, y_star, x0, cfg, num_probes_eval=2)
        out_minus = solve_dipg(model_minus, y_star, x0, cfg, num_probes_eval=2)

        rmse_plus = rmse_x(out_plus["x_hat"], x_true)
        rmse_minus = rmse_x(out_minus["x_hat"], x_true)

        rows.append({
            "idx": i,
            "rmse_plus": rmse_plus,
            "rmse_minus": rmse_minus,
            "gap_minus_minus_plus": rmse_minus - rmse_plus,
        })

    df = pd.DataFrame(rows).sort_values("gap_minus_minus_plus", ascending=False)
    return df.head(num_cases)["idx"].tolist()

selected_case_ids = pick_representative_cases(
    data["x_test"], data["y_noisy_test_norm"], model_plus_jcp, model_minus_jcp, cfg, num_cases=3
)
selected_case_ids

In [ ]:
def recover_case(case_idx: int, method_name: str):
    x_true = data["x_test"][case_idx:case_idx+1]
    y_star = data["y_noisy_test_norm"][case_idx:case_idx+1].to(device)
    x0 = torch.zeros_like(x_true).to(device)

    if method_name == "D-IPG (+JCP)":
        out = solve_dipg(model_plus_jcp, y_star, x0, cfg)
    elif method_name == "D-IPG (-JCP)":
        out = solve_dipg(model_minus_jcp, y_star, x0, cfg)
    elif method_name == "Gradient Descent":
        out = solve_gradient_descent(model_plus_jcp.f, y_star, x0, cfg)
    elif method_name == "GN":
        out = solve_gauss_newton(model_plus_jcp.f, y_star, x0, cfg)
    elif method_name == "LM":
        out = solve_lm(model_plus_jcp.f, y_star, x0, cfg)
    elif method_name == "L-BFGS":
        out = solve_lbfgs(model_plus_jcp.f, y_star, x0, cfg)
    else:
        raise ValueError(method_name)

    return {
        "true_x": data["x_test"][case_idx, 0].cpu().numpy(),
        "obs_y": data["y_noisy_test"][case_idx, 0].cpu().numpy(),
        "recovered_x": out["x_hat"][0, 0].cpu().numpy(),
    }

In [ ]:
methods_for_qual = ["D-IPG (+JCP)", "D-IPG (-JCP)", "GN", "LM", "L-BFGS"]

n_cases = len(selected_case_ids)
n_cols = 2 + len(methods_for_qual)

fig, axes = plt.subplots(n_cases, n_cols, figsize=(2.5 * n_cols, 2.5 * n_cases), dpi=170)

if n_cases == 1:
    axes = np.expand_dims(axes, axis=0)

for r, idx in enumerate(selected_case_ids):
    rec_plus = recover_case(idx, "D-IPG (+JCP)")
    rec_minus = recover_case(idx, "D-IPG (-JCP)")
    rec_gn = recover_case(idx, "GN")
    rec_lm = recover_case(idx, "LM")
    rec_lbfgs = recover_case(idx, "L-BFGS")

    axes[r, 0].imshow(rec_plus["true_x"], cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[r, 0].set_title("True $u_0$" if r == 0 else "")
    axes[r, 0].axis("off")

    axes[r, 1].imshow(rec_plus["obs_y"], cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[r, 1].set_title("Observed $u_T$" if r == 0 else "")
    axes[r, 1].axis("off")

    qual_maps = [
        rec_plus["recovered_x"],
        rec_minus["recovered_x"],
        rec_gn["recovered_x"],
        rec_lm["recovered_x"],
        rec_lbfgs["recovered_x"],
    ]

    for j, qmap in enumerate(qual_maps):
        axes[r, 2 + j].imshow(qmap, cmap="coolwarm", vmin=-1.0, vmax=1.0)
        axes[r, 2 + j].set_title(methods_for_qual[j] if r == 0 else "")
        axes[r, 2 + j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plot_df = main_results.copy()
plot_df["label"] = plot_df["ablation"] + " | " + plot_df["method"]

plt.figure(figsize=(8.2, 4.5), dpi=170)
plt.bar(plot_df["label"], plot_df["mean_rmse_x"])
plt.ylabel("Mean RMSE$_x$")
plt.title("Allen–Cahn: mean reconstruction error")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
rjcp_plot_df = rjcp_summary.copy()

plt.figure(figsize=(5, 4.2), dpi=170)
plt.bar(rjcp_plot_df["ablation"], rjcp_plot_df["test_rjcp"])
plt.ylabel("Test RJCP")
plt.title("Inverse-geometry comparison")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6.2, 4.2), dpi=170)
plt.plot(history_reverse["epoch"], history_reverse["val_reconstruction"], label="Reverse-only")
plt.plot(history_reverse_jcp_plus["epoch"], history_reverse_jcp_plus["val_reconstruction"], label="+JCP")
plt.plot(history_reverse_jcp_minus["epoch"], history_reverse_jcp_minus["val_reconstruction"], label="-JCP")
plt.xlabel("Epoch")
plt.ylabel("Validation reconstruction")
plt.title("Validation reconstruction during reverse training")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6.2, 4.2), dpi=170)
plt.plot(history_reverse["epoch"], history_reverse["val_rjcp"], label="Reverse-only")
plt.plot(history_reverse_jcp_plus["epoch"], history_reverse_jcp_plus["val_rjcp"], label="+JCP")
plt.plot(history_reverse_jcp_minus["epoch"], history_reverse_jcp_minus["val_rjcp"], label="-JCP")
plt.xlabel("Epoch")
plt.ylabel("Validation RJCP")
plt.title("Validation RJCP during reverse training")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
conv_case_idx = selected_case_ids[0]

x_true_case = data["x_test"][conv_case_idx:conv_case_idx+1]
y_star_case = data["y_noisy_test_norm"][conv_case_idx:conv_case_idx+1].to(device)
x0_case = torch.zeros_like(x_true_case).to(device)

trace_plus = solve_dipg_trace(model_plus_jcp, y_star_case, x0_case, cfg, x_true=x_true_case, num_probes_eval=2)
trace_minus = solve_dipg_trace(model_minus_jcp, y_star_case, x0_case, cfg, x_true=x_true_case, num_probes_eval=2)
trace_gd = solve_gd_trace(model_plus_jcp.f, y_star_case, x0_case, cfg, x_true=x_true_case)
trace_gn = solve_gn_trace(model_plus_jcp.f, y_star_case, x0_case, cfg, x_true=x_true_case)
trace_lm = solve_lm_trace(model_plus_jcp.f, y_star_case, x0_case, cfg, x_true=x_true_case)
trace_lbfgs = solve_lbfgs_trace(model_plus_jcp.f, y_star_case, x0_case, cfg, x_true=x_true_case)

In [ ]:
plt.figure(figsize=(6.7, 4.5), dpi=170)
plt.plot(trace_plus["rmse_trace"], marker="o", ms=3, label="D-IPG (+JCP)")
plt.plot(trace_minus["rmse_trace"], marker="o", ms=3, label="D-IPG (-JCP)")
plt.plot(trace_gd["rmse_trace"], marker="o", ms=3, label="Gradient Descent")
plt.plot(trace_gn["rmse_trace"], marker="o", ms=3, label="GN")
plt.plot(trace_lm["rmse_trace"], marker="o", ms=3, label="LM")
plt.plot(trace_lbfgs["rmse_trace"], marker="o", ms=3, label="L-BFGS")
plt.xlabel("Iteration")
plt.ylabel("RMSE$_x$")
plt.title(f"Convergence on test case {conv_case_idx}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6.7, 4.5), dpi=170)
plt.plot(trace_plus["residual_trace"], marker="o", ms=3, label="D-IPG (+JCP)")
plt.plot(trace_minus["residual_trace"], marker="o", ms=3, label="D-IPG (-JCP)")
plt.plot(trace_gd["residual_trace"], marker="o", ms=3, label="Gradient Descent")
plt.plot(trace_gn["residual_trace"], marker="o", ms=3, label="GN")
plt.plot(trace_lm["residual_trace"], marker="o", ms=3, label="LM")
plt.plot(trace_lbfgs["residual_trace"], marker="o", ms=3, label="L-BFGS")
plt.xlabel("Iteration")
plt.ylabel("Residual")
plt.title(f"Residual convergence on test case {conv_case_idx}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7), dpi=170)

axes[0, 0].plot(trace_plus["alpha_accept_trace"], marker="o", ms=3, label="+JCP")
axes[0, 0].plot(trace_minus["alpha_accept_trace"], marker="o", ms=3, label="-JCP")
axes[0, 0].set_title("Accepted step size")
axes[0, 0].set_xlabel("Accepted step")
axes[0, 0].legend()

axes[0, 1].plot(trace_plus["step_norm_accept_trace"], marker="o", ms=3, label="+JCP")
axes[0, 1].plot(trace_minus["step_norm_accept_trace"], marker="o", ms=3, label="-JCP")
axes[0, 1].set_title("Accepted step norm")
axes[0, 1].set_xlabel("Accepted step")
axes[0, 1].legend()

axes[1, 0].plot(trace_plus["cosine_accept_trace"], marker="o", ms=3, label="+JCP")
axes[1, 0].plot(trace_minus["cosine_accept_trace"], marker="o", ms=3, label="-JCP")
axes[1, 0].set_title("Cosine with negative gradient")
axes[1, 0].set_xlabel("Accepted step")
axes[1, 0].legend()

axes[1, 1].plot(trace_plus["residual_drop_accept_trace"], marker="o", ms=3, label="+JCP")
axes[1, 1].plot(trace_minus["residual_drop_accept_trace"], marker="o", ms=3, label="-JCP")
axes[1, 1].set_title("Residual drop per accepted step")
axes[1, 1].set_xlabel("Accepted step")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
FIG_DIR = "/content/allen_cahn_figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# qualitative
n_cases = len(selected_case_ids)
n_cols = 2 + len(methods_for_qual)

fig, axes = plt.subplots(n_cases, n_cols, figsize=(2.5 * n_cols, 2.5 * n_cases), dpi=220)
if n_cases == 1:
    axes = np.expand_dims(axes, axis=0)

for r, idx in enumerate(selected_case_ids):
    rec_plus = recover_case(idx, "D-IPG (+JCP)")
    rec_minus = recover_case(idx, "D-IPG (-JCP)")
    rec_gn = recover_case(idx, "GN")
    rec_lm = recover_case(idx, "LM")
    rec_lbfgs = recover_case(idx, "L-BFGS")

    axes[r, 0].imshow(rec_plus["true_x"], cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[r, 0].set_title("True $u_0$" if r == 0 else "")
    axes[r, 0].axis("off")

    axes[r, 1].imshow(rec_plus["obs_y"], cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[r, 1].set_title("Observed $u_T$" if r == 0 else "")
    axes[r, 1].axis("off")

    qual_maps = [
        rec_plus["recovered_x"],
        rec_minus["recovered_x"],
        rec_gn["recovered_x"],
        rec_lm["recovered_x"],
        rec_lbfgs["recovered_x"],
    ]

    for j, qmap in enumerate(qual_maps):
        axes[r, 2 + j].imshow(qmap, cmap="coolwarm", vmin=-1.0, vmax=1.0)
        axes[r, 2 + j].set_title(methods_for_qual[j] if r == 0 else "")
        axes[r, 2 + j].axis("off")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_qualitative_grid.pdf", bbox_inches="tight")
plt.close(fig)

# rmse bar
fig = plt.figure(figsize=(8.2, 4.5), dpi=220)
plt.bar(plot_df["label"], plot_df["mean_rmse_x"])
plt.ylabel("Mean RMSE$_x$")
plt.title("Allen–Cahn: mean reconstruction error")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_mean_rmse_bar.pdf", bbox_inches="tight")
plt.close(fig)

# rjcp bar
fig = plt.figure(figsize=(5, 4.2), dpi=220)
plt.bar(rjcp_plot_df["ablation"], rjcp_plot_df["test_rjcp"])
plt.ylabel("Test RJCP")
plt.title("Inverse-geometry comparison")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_rjcp_bar.pdf", bbox_inches="tight")
plt.close(fig)

# val reconstruction
fig = plt.figure(figsize=(6.2, 4.2), dpi=220)
plt.plot(history_reverse["epoch"], history_reverse["val_reconstruction"], label="Reverse-only")
plt.plot(history_reverse_jcp_plus["epoch"], history_reverse_jcp_plus["val_reconstruction"], label="+JCP")
plt.plot(history_reverse_jcp_minus["epoch"], history_reverse_jcp_minus["val_reconstruction"], label="-JCP")
plt.xlabel("Epoch")
plt.ylabel("Validation reconstruction")
plt.title("Validation reconstruction during reverse training")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_val_reconstruction_curve.pdf", bbox_inches="tight")
plt.close(fig)

# val rjcp
fig = plt.figure(figsize=(6.2, 4.2), dpi=220)
plt.plot(history_reverse["epoch"], history_reverse["val_rjcp"], label="Reverse-only")
plt.plot(history_reverse_jcp_plus["epoch"], history_reverse_jcp_plus["val_rjcp"], label="+JCP")
plt.plot(history_reverse_jcp_minus["epoch"], history_reverse_jcp_minus["val_rjcp"], label="-JCP")
plt.xlabel("Epoch")
plt.ylabel("Validation RJCP")
plt.title("Validation RJCP during reverse training")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_val_rjcp_curve.pdf", bbox_inches="tight")
plt.close(fig)

# convergence rmse
fig = plt.figure(figsize=(6.7, 4.5), dpi=220)
plt.plot(trace_plus["rmse_trace"], marker="o", ms=3, label="D-IPG (+JCP)")
plt.plot(trace_minus["rmse_trace"], marker="o", ms=3, label="D-IPG (-JCP)")
plt.plot(trace_gd["rmse_trace"], marker="o", ms=3, label="Gradient Descent")
plt.plot(trace_gn["rmse_trace"], marker="o", ms=3, label="GN")
plt.plot(trace_lm["rmse_trace"], marker="o", ms=3, label="LM")
plt.plot(trace_lbfgs["rmse_trace"], marker="o", ms=3, label="L-BFGS")
plt.xlabel("Iteration")
plt.ylabel("RMSE$_x$")
plt.title(f"Convergence on test case {conv_case_idx}")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_convergence_rmse.pdf", bbox_inches="tight")
plt.close(fig)

# convergence residual
fig = plt.figure(figsize=(6.7, 4.5), dpi=220)
plt.plot(trace_plus["residual_trace"], marker="o", ms=3, label="D-IPG (+JCP)")
plt.plot(trace_minus["residual_trace"], marker="o", ms=3, label="D-IPG (-JCP)")
plt.plot(trace_gd["residual_trace"], marker="o", ms=3, label="Gradient Descent")
plt.plot(trace_gn["residual_trace"], marker="o", ms=3, label="GN")
plt.plot(trace_lm["residual_trace"], marker="o", ms=3, label="LM")
plt.plot(trace_lbfgs["residual_trace"], marker="o", ms=3, label="L-BFGS")
plt.xlabel("Iteration")
plt.ylabel("Residual")
plt.title(f"Residual convergence on test case {conv_case_idx}")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_convergence_residual.pdf", bbox_inches="tight")
plt.close(fig)

# dipg diagnostics
fig, axes = plt.subplots(2, 2, figsize=(9, 7), dpi=220)
axes[0, 0].plot(trace_plus["alpha_accept_trace"], marker="o", ms=3, label="+JCP")
axes[0, 0].plot(trace_minus["alpha_accept_trace"], marker="o", ms=3, label="-JCP")
axes[0, 0].set_title("Accepted step size")
axes[0, 0].set_xlabel("Accepted step")
axes[0, 0].legend()

axes[0, 1].plot(trace_plus["step_norm_accept_trace"], marker="o", ms=3, label="+JCP")
axes[0, 1].plot(trace_minus["step_norm_accept_trace"], marker="o", ms=3, label="-JCP")
axes[0, 1].set_title("Accepted step norm")
axes[0, 1].set_xlabel("Accepted step")
axes[0, 1].legend()

axes[1, 0].plot(trace_plus["cosine_accept_trace"], marker="o", ms=3, label="+JCP")
axes[1, 0].plot(trace_minus["cosine_accept_trace"], marker="o", ms=3, label="-JCP")
axes[1, 0].set_title("Cosine with negative gradient")
axes[1, 0].set_xlabel("Accepted step")
axes[1, 0].legend()

axes[1, 1].plot(trace_plus["residual_drop_accept_trace"], marker="o", ms=3, label="+JCP")
axes[1, 1].plot(trace_minus["residual_drop_accept_trace"], marker="o", ms=3, label="-JCP")
axes[1, 1].set_title("Residual drop per accepted step")
axes[1, 1].set_xlabel("Accepted step")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/allen_cahn_dipg_diagnostics.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
def plot_allen_cahn_interface_overlay(true_field, rec_field, title=""):
    plt.figure(figsize=(4.5, 4.2), dpi=170)

    plt.imshow(true_field, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    plt.contour(true_field, levels=[0.0], colors="black", linewidths=2.0)
    plt.contour(rec_field, levels=[0.0], colors="lime", linewidths=1.8, linestyles="--")

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/allen_cahn_comparison_DIPG_+JCP.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
idx = selected_case_ids[0]
rec = recover_case(idx, "D-IPG (+JCP)")
plot_allen_cahn_interface_overlay(
    rec["true_x"],
    rec["recovered_x"],
    title="Allen–Cahn interface comparison: D-IPG (+JCP)"
)

In [ ]:
def plot_allen_cahn_error_map(true_field, rec_field, title=""):
    err = rec_field - true_field
    vmax = np.max(np.abs(err)) + 1e-8

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), dpi=170)

    axes[0].imshow(true_field, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_title("True")
    axes[0].axis("off")

    axes[1].imshow(rec_field, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[1].set_title("Reconstruction")
    axes[1].axis("off")

    im = axes[2].imshow(err, cmap="bwr", vmin=-vmax, vmax=vmax)
    axes[2].set_title("Error")
    axes[2].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/allen_cahn_error_visualization_+JCP.pdf", bbox_inches="tight")

    plt.show()

In [ ]:
idx = selected_case_ids[0]
rec = recover_case(idx, "D-IPG (+JCP)")
plot_allen_cahn_error_map(
    rec["true_x"],
    rec["recovered_x"],
    title="Allen–Cahn error visualization: D-IPG (+JCP)"
)

In [ ]:
def plot_allen_cahn_triptych(case_idx: int, method_name: str):
    rec = recover_case(case_idx, method_name)

    true_field = rec["true_x"]
    pred_field = rec["recovered_x"]
    abs_err = np.abs(pred_field - true_field)

    vmax_err = abs_err.max() + 1e-12

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), dpi=180)

    im0 = axes[0].imshow(true_field, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_title("Exact")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(pred_field, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[1].set_title(method_name)
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(abs_err, cmap="viridis", vmin=0.0, vmax=vmax_err)
    axes[2].set_title("Absolute error")
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/allen_cahn_error_all.pdf", bbox_inches="tight")

    plt.show()

In [ ]:
plot_allen_cahn_triptych(selected_case_ids[0], "D-IPG (+JCP)")
plot_allen_cahn_triptych(selected_case_ids[0], "D-IPG (-JCP)")
plot_allen_cahn_triptych(selected_case_ids[0], "L-BFGS")

In [ ]:
def plot_allen_cahn_interface_comparison(case_idx: int, methods=("D-IPG (+JCP)", "D-IPG (-JCP)", "L-BFGS")):
    rec_true = recover_case(case_idx, methods[0])
    true_field = rec_true["true_x"]

    fig, axes = plt.subplots(1, len(methods), figsize=(4.2 * len(methods), 4.0), dpi=180)

    if len(methods) == 1:
        axes = [axes]

    for ax, method_name in zip(axes, methods):
        rec = recover_case(case_idx, method_name)
        pred_field = rec["recovered_x"]

        ax.imshow(true_field, cmap="coolwarm", vmin=-1.0, vmax=1.0)
        ax.contour(true_field, levels=[0.0], colors="black", linewidths=2.0)
        ax.contour(pred_field, levels=[0.0], colors="lime", linewidths=1.8, linestyles="--")
        ax.set_title(method_name)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/allen_cahn_DIPG_LFBGS.pdf", bbox_inches="tight")

    plt.show()

In [ ]:
plot_allen_cahn_interface_comparison(selected_case_ids[0])

##Seed Sweep

In [ ]:
SEED_LIST = [3, 7, 11]
SEED_LIST

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def run_full_allen_cahn_experiment_for_seed(seed: int, base_cfg: AllenCahn2DConfig):
    print(f"\nSeed {seed}")
    set_all_seeds(seed)

    cfg_local = AllenCahn2DConfig(**asdict(base_cfg))
    data_local = build_dataset(cfg_local)

    forward_train_loader_local = DataLoader(
        TensorDataset(data_local["x_train"], data_local["y_clean_train_norm"]),
        batch_size=cfg_local.batch_size,
        shuffle=True,
        drop_last=False,
    )
    forward_val_loader_local = DataLoader(
        TensorDataset(data_local["x_val"], data_local["y_clean_val_norm"]),
        batch_size=cfg_local.batch_size,
        shuffle=False,
        drop_last=False,
    )

    reverse_train_loader_local = DataLoader(
        TensorDataset(data_local["x_train"], data_local["y_clean_train_norm"]),
        batch_size=cfg_local.batch_size,
        shuffle=True,
        drop_last=False,
    )
    reverse_val_loader_local = DataLoader(
        TensorDataset(data_local["x_val"], data_local["y_clean_val_norm"]),
        batch_size=cfg_local.batch_size,
        shuffle=False,
        drop_last=False,
    )

    reverse_jcp_train_loader_local = DataLoader(
        TensorDataset(data_local["x_train"], data_local["y_clean_train_norm"]),
        batch_size=cfg_local.batch_size,
        shuffle=True,
        drop_last=False,
    )
    reverse_jcp_val_loader_local = DataLoader(
        TensorDataset(data_local["x_val"], data_local["y_clean_val_norm"]),
        batch_size=cfg_local.batch_size,
        shuffle=False,
        drop_last=False,
    )

    def estimate_rjcp_dataset_local(model, x_data, num_probes=2, max_samples=24):
        model.eval()
        x_subset = x_data[:max_samples].to(device)
        vals = []
        for i in range(x_subset.shape[0]):
            vals.append(single_sample_probe_jcp(model, x_subset[i], num_probes=num_probes).detach().cpu())
        return float(torch.stack(vals).mean().item())

    @torch.no_grad()
    def evaluate_forward_validation_loss_local(model, loader):
        model.eval()
        total = 0.0
        count = 0
        for x_cpu, y_cpu in loader:
            x = x_cpu.to(device)
            y = y_cpu.to(device)
            y_pred = model.f(x)
            total += float(F.mse_loss(y_pred, y, reduction="sum").item())
            count += y.numel()
        return total / max(count, 1)

    @torch.no_grad()
    def evaluate_reverse_validation_local(model, loader):
        model.eval()
        totals = {"reconstruction": 0.0, "cycle": 0.0, "loss": 0.0}
        count = 0

        for x_cpu, y_cpu in loader:
            x = x_cpu.to(device)
            y = y_cpu.to(device)

            x_hat = model.g(y)
            y_cyc = model.f(x_hat)

            rec_loss = F.mse_loss(x_hat, x)
            cyc_loss = F.mse_loss(y_cyc, y)
            loss = cfg_local.reconstruction_weight * rec_loss + cfg_local.cycle_weight * cyc_loss

            bs = x.shape[0]
            totals["reconstruction"] += float(rec_loss.item()) * bs
            totals["cycle"] += float(cyc_loss.item()) * bs
            totals["loss"] += float(loss.item()) * bs
            count += bs

        return {k: v / count for k, v in totals.items()}

    def train_forward_map_only_local(train_loader, val_loader):
        model = DeceptronAllenCahnModel(
            cfg_local.x_channels,
            cfg_local.y_channels,
            cfg_local.hidden_channels,
            cfg_local.num_res_blocks,
            cfg_local.negative_slope,
        ).to(device)

        optimizer = torch.optim.Adam(
            model.forward_map.parameters(),
            lr=cfg_local.forward_lr,
            weight_decay=cfg_local.forward_weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg_local.forward_epochs)

        history = []
        best_state = None
        best_val = float("inf")

        for epoch in range(cfg_local.forward_epochs):
            model.train()
            train_total = 0.0
            count = 0

            for x_cpu, y_cpu in train_loader:
                x = x_cpu.to(device)
                y = y_cpu.to(device)

                optimizer.zero_grad(set_to_none=True)
                loss = F.mse_loss(model.f(x), y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.forward_map.parameters(), cfg_local.gradient_clip)
                optimizer.step()

                train_total += float(loss.item()) * x.shape[0]
                count += x.shape[0]

            scheduler.step()

            val_task = evaluate_forward_validation_loss_local(model, val_loader)
            val_rjcp = estimate_rjcp_dataset_local(
                model,
                data_local["x_val"],
                cfg_local.jcp_num_probes_eval,
                cfg_local.eval_subset_rjcp,
            )

            history.append({
                "epoch": epoch + 1,
                "train_task": train_total / count,
                "val_task": val_task,
                "val_rjcp": val_rjcp,
            })

            if val_task < best_val:
                best_val = val_task
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        return model, pd.DataFrame(history)

    def train_reverse_map_only_local(model_init, train_loader, val_loader):
        model = DeceptronAllenCahnModel(
            cfg_local.x_channels,
            cfg_local.y_channels,
            cfg_local.hidden_channels,
            cfg_local.num_res_blocks,
            cfg_local.negative_slope,
        ).to(device)

        model.forward_map.load_state_dict({k: v.detach().clone() for k, v in model_init.forward_map.state_dict().items()})
        freeze_forward_map(model)

        optimizer = torch.optim.Adam(
            model.reverse_map.parameters(),
            lr=cfg_local.reverse_lr,
            weight_decay=cfg_local.reverse_weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg_local.reverse_epochs)

        history = []
        best_state = None
        best_score = float("inf")

        for epoch in range(cfg_local.reverse_epochs):
            model.train()
            totals = {"loss": 0.0, "reconstruction": 0.0, "cycle": 0.0}
            count = 0

            for x_cpu, y_cpu in train_loader:
                x = x_cpu.to(device)
                y = y_cpu.to(device)

                optimizer.zero_grad(set_to_none=True)

                x_hat = model.g(y)
                y_tilde = y + cfg_local.y_tilde_noise * torch.randn_like(y)
                y_cyc = model.f(model.g(y_tilde))

                rec_loss = F.mse_loss(x_hat, x)
                cyc_loss = F.mse_loss(y_cyc, y_tilde)
                loss = cfg_local.reconstruction_weight * rec_loss + cfg_local.cycle_weight * cyc_loss

                loss.backward()
                nn.utils.clip_grad_norm_(model.reverse_map.parameters(), cfg_local.gradient_clip)
                optimizer.step()

                bs = x.shape[0]
                count += bs
                totals["loss"] += float(loss.item()) * bs
                totals["reconstruction"] += float(rec_loss.item()) * bs
                totals["cycle"] += float(cyc_loss.item()) * bs

            scheduler.step()

            val_stats = evaluate_reverse_validation_local(model, val_loader)
            val_rjcp = estimate_rjcp_dataset_local(
                model,
                data_local["x_val"],
                cfg_local.jcp_num_probes_eval,
                cfg_local.eval_subset_rjcp,
            )

            row = {
                "epoch": epoch + 1,
                "val_loss": val_stats["loss"],
                "val_reconstruction": val_stats["reconstruction"],
                "val_rjcp": val_rjcp,
            }
            history.append(row)

            score = row["val_reconstruction"] + 0.05 * row["val_rjcp"]
            if score < best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        return model, pd.DataFrame(history)

    def train_reverse_map_with_jcp_fixed_forward_local(model_init, train_loader, val_loader, use_jcp, reference_reconstruction):
        model = DeceptronAllenCahnModel(
            cfg_local.x_channels,
            cfg_local.y_channels,
            cfg_local.hidden_channels,
            cfg_local.num_res_blocks,
            cfg_local.negative_slope,
        ).to(device)

        model.forward_map.load_state_dict({k: v.detach().clone() for k, v in model_init.forward_map.state_dict().items()})
        model.reverse_map.load_state_dict({k: v.detach().clone() for k, v in model_init.reverse_map.state_dict().items()})
        freeze_forward_map(model)

        optimizer = torch.optim.Adam(
            model.reverse_map.parameters(),
            lr=cfg_local.reverse_jcp_lr,
            weight_decay=cfg_local.reverse_jcp_weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg_local.reverse_jcp_epochs)

        history = []
        best_state = None
        best_score = float("inf")

        for epoch in range(cfg_local.reverse_jcp_epochs):
            model.train()
            totals = {"loss": 0.0, "reconstruction": 0.0, "cycle": 0.0, "probe_jcp": 0.0}
            count = 0

            for x_cpu, y_cpu in train_loader:
                x = x_cpu.to(device)
                y = y_cpu.to(device)

                optimizer.zero_grad(set_to_none=True)

                x_hat = model.g(y)
                y_tilde = y + cfg_local.y_tilde_noise * torch.randn_like(y)
                y_cyc = model.f(model.g(y_tilde))

                rec_loss = F.mse_loss(x_hat, x)
                cyc_loss = F.mse_loss(y_cyc, y_tilde)

                if use_jcp:
                    bs_jcp = min(cfg_local.jcp_batch_subsample, x.shape[0])
                    y_subset = y[:bs_jcp]
                    with torch.no_grad():
                        x_pred_subset = model.g(y_subset)

                    probe_jcp_loss = cfg_local.probe_jcp_weight * batch_probe_jcp(
                        model,
                        x_pred_subset,
                        num_probes=cfg_local.jcp_num_probes_train,
                    )
                else:
                    probe_jcp_loss = torch.tensor(0.0, device=device)

                loss = cfg_local.reconstruction_weight * rec_loss + cfg_local.cycle_weight * cyc_loss + probe_jcp_loss

                loss.backward()
                nn.utils.clip_grad_norm_(model.reverse_map.parameters(), cfg_local.gradient_clip)
                optimizer.step()

                bs = x.shape[0]
                count += bs
                totals["loss"] += float(loss.item()) * bs
                totals["reconstruction"] += float(rec_loss.item()) * bs
                totals["cycle"] += float(cyc_loss.item()) * bs
                totals["probe_jcp"] += float(probe_jcp_loss.item()) * bs

            scheduler.step()

            val_stats = evaluate_reverse_validation_local(model, val_loader)
            val_rjcp = estimate_rjcp_dataset_local(
                model,
                data_local["x_val"],
                cfg_local.jcp_num_probes_eval,
                cfg_local.eval_subset_rjcp,
            )

            row = {
                "epoch": epoch + 1,
                "val_loss": val_stats["loss"],
                "val_reconstruction": val_stats["reconstruction"],
                "val_rjcp": val_rjcp,
            }
            history.append(row)

            recon_ratio = row["val_reconstruction"] / max(reference_reconstruction, 1e-12)
            if recon_ratio <= cfg_local.recon_guard_factor:
                score = row["val_rjcp"] + 1e-3 * row["val_loss"]
            else:
                score = row["val_rjcp"] + 1000.0 * (recon_ratio - cfg_local.recon_guard_factor)

            if score < best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        return model, pd.DataFrame(history)

    model_stage1_local, history_forward_local = train_forward_map_only_local(
        forward_train_loader_local,
        forward_val_loader_local,
    )

    model_stage2_local, history_reverse_local = train_reverse_map_only_local(
        model_stage1_local,
        reverse_train_loader_local,
        reverse_val_loader_local,
    )

    reference_recon = float(history_reverse_local["val_reconstruction"].min())

    model_plus_local, history_plus_local = train_reverse_map_with_jcp_fixed_forward_local(
        model_stage2_local,
        reverse_jcp_train_loader_local,
        reverse_jcp_val_loader_local,
        use_jcp=True,
        reference_reconstruction=reference_recon,
    )

    model_minus_local, history_minus_local = train_reverse_map_with_jcp_fixed_forward_local(
        model_stage2_local,
        reverse_jcp_train_loader_local,
        reverse_jcp_val_loader_local,
        use_jcp=False,
        reference_reconstruction=reference_recon,
    )

    main_results_local = evaluate_allen_cahn_methods(
        data_local["x_test"],
        data_local["y_noisy_test_norm"],
        model_plus_local,
        model_minus_local,
        cfg_local,
    )

    rjcp_summary_local = pd.DataFrame([
        {
            "ablation": "+JCP",
            "val_reconstruction": evaluate_reverse_validation_local(model_plus_local, reverse_jcp_val_loader_local)["reconstruction"],
            "val_rjcp": estimate_rjcp_dataset_local(model_plus_local, data_local["x_val"], cfg_local.jcp_num_probes_eval, cfg_local.eval_subset_rjcp),
            "test_rjcp": estimate_rjcp_dataset_local(model_plus_local, data_local["x_test"], cfg_local.jcp_num_probes_eval, cfg_local.eval_subset_rjcp),
        },
        {
            "ablation": "-JCP",
            "val_reconstruction": evaluate_reverse_validation_local(model_minus_local, reverse_jcp_val_loader_local)["reconstruction"],
            "val_rjcp": estimate_rjcp_dataset_local(model_minus_local, data_local["x_val"], cfg_local.jcp_num_probes_eval, cfg_local.eval_subset_rjcp),
            "test_rjcp": estimate_rjcp_dataset_local(model_minus_local, data_local["x_test"], cfg_local.jcp_num_probes_eval, cfg_local.eval_subset_rjcp),
        },
    ])

    main_results_local["seed"] = seed
    rjcp_summary_local["seed"] = seed

    return {
        "seed": seed,
        "main_results": main_results_local,
        "rjcp_summary": rjcp_summary_local,
        "history_forward": history_forward_local,
        "history_reverse": history_reverse_local,
        "history_plus": history_plus_local,
        "history_minus": history_minus_local,
    }

In [ ]:
seed_runs = []

for seed in SEED_LIST:
    out = run_full_allen_cahn_experiment_for_seed(seed, cfg)
    seed_runs.append(out)

In [ ]:
seed_main_results = pd.concat([run["main_results"] for run in seed_runs], ignore_index=True)
seed_rjcp_results = pd.concat([run["rjcp_summary"] for run in seed_runs], ignore_index=True)

print(seed_main_results.round(6).to_string(index=False))
print()
print(seed_rjcp_results.round(6).to_string(index=False))

In [ ]:
seed_main_summary = (
    seed_main_results
    .groupby(["ablation", "method"], as_index=False)
    .agg(
        mean_success_rate=("success_rate", "mean"),
        std_success_rate=("success_rate", "std"),
        mean_mean_iters=("mean_iters", "mean"),
        std_mean_iters=("mean_iters", "std"),
        mean_mean_rmse_x=("mean_rmse_x", "mean"),
        std_mean_rmse_x=("mean_rmse_x", "std"),
        mean_mean_final_residual=("mean_final_residual", "mean"),
        std_mean_final_residual=("mean_final_residual", "std"),
        mean_mean_time_sec=("mean_time_sec", "mean"),
        std_mean_time_sec=("mean_time_sec", "std"),
        mean_mean_final_rjcp=("mean_final_rjcp", "mean"),
        std_mean_final_rjcp=("mean_final_rjcp", "std"),
    )
)

print(seed_main_summary.round(6).to_string(index=False))

In [ ]:
seed_rjcp_summary = (
    seed_rjcp_results
    .groupby(["ablation"], as_index=False)
    .agg(
        mean_val_reconstruction=("val_reconstruction", "mean"),
        std_val_reconstruction=("val_reconstruction", "std"),
        mean_val_rjcp=("val_rjcp", "mean"),
        std_val_rjcp=("val_rjcp", "std"),
        mean_test_rjcp=("test_rjcp", "mean"),
        std_test_rjcp=("test_rjcp", "std"),
    )
)

print(seed_rjcp_summary.round(6).to_string(index=False))

In [ ]:
def mean_pm_std(mean_val, std_val, digits=4):
    if pd.isna(std_val):
        return f"{mean_val:.{digits}f}"
    return f"{mean_val:.{digits}f} ± {std_val:.{digits}f}"

compact_rows = []
for _, row in seed_main_summary.iterrows():
    compact_rows.append({
        "ablation": row["ablation"],
        "method": row["method"],
        "success_rate": mean_pm_std(row["mean_success_rate"], row["std_success_rate"], 4),
        "mean_rmse_x": mean_pm_std(row["mean_mean_rmse_x"], row["std_mean_rmse_x"], 4),
        "mean_final_residual": mean_pm_std(row["mean_mean_final_residual"], row["std_mean_final_residual"], 4),
        "mean_time_sec": mean_pm_std(row["mean_mean_time_sec"], row["std_mean_time_sec"], 4),
        "mean_final_rjcp": mean_pm_std(row["mean_mean_final_rjcp"], row["std_mean_final_rjcp"], 4),
    })

compact_seed_table = pd.DataFrame(compact_rows)
print(compact_seed_table.to_string(index=False))

In [ ]:
seed_main_results.to_csv("/content/allen_cahn_seed_main_results.csv", index=False)
seed_rjcp_results.to_csv("/content/allen_cahn_seed_rjcp_results.csv", index=False)
seed_main_summary.to_csv("/content/allen_cahn_seed_main_summary.csv", index=False)
seed_rjcp_summary.to_csv("/content/allen_cahn_seed_rjcp_summary.csv", index=False)